# CerberusVision Phase 5 - Qwen-2.5-7B-Instruct (From Scratch QLoRA)

Bu defter, veri sızıntısını (data leakage) engellemek ve ezberlemeyi kırmak için modern TRL SFTConfig altyapısıyla güncellenmiştir.

**Kritik Özellikler:**
- **Aile bazlı Train-Val ayrımı**: Her `shipping_instruction_reference` bir belge ailesidir. Aynı ailenin TÜM varyantları (farklı booking/konteyner numaralı olsa bile) tek tarafta kalır.
- Validation seti, eğitimde HİÇ görülmemiş yeni bir belge şablonundan oluşur → gerçek genelleme ölçümü.
- Dinamik BF16 tespiti.
- TRL SFTConfig modern API kullanımı.
- Model checkpointing (load_best_model_at_end).

**Güncel Veri (2026-07-24):**
- Train: 1043 örnek (10 belge ailesi)
- Validation: 75 örnek (1 belge ailesi — SI-EXP-4422-2026)
- Sıfır aile çakışması (family leakage = 0)
- Sıfır exact duplicate


In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|endoftext|>'})

print("PAD Token ID:", tokenizer.pad_token_id)
print("EOS Token ID:", tokenizer.eos_token_id)
assert tokenizer.pad_token_id != tokenizer.eos_token_id
assert tokenizer.eos_token == "<|im_end|>"


In [ ]:
train_dataset = load_dataset("json", data_files="phase5_train.jsonl", split="train")
eval_dataset = load_dataset("json", data_files="phase5_validation.jsonl", split="train")

def format_dataset(example):
    return {
        "prompt": [
            {"role": "system", "content": "Extract shipping instruction data from OCR text as JSON."},
            {"role": "user", "content": str(example['input'])}
        ],
        "completion": [
            {"role": "assistant", "content": str(example['output'])}
        ]
    }

train_dataset = train_dataset.map(format_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_dataset, remove_columns=eval_dataset.column_names)
print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

training_args = SFTConfig(
    output_dir="./qwen-2.5-7b-phase5-lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_32bit",
    seed=42,
    data_seed=42,
    report_to="none",
    max_length=2048,
    completion_only_loss=True,
    eos_token="<|im_end|>",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained("adapter_best")
tokenizer.save_pretrained("adapter_best")
